# A2A Protocol: Deep Dive using Google's Official A2A SDK

This notebook demonstrates the **Agent-to-Agent (A2A) Protocol** using Google's official Python SDK. A2A is an open protocol under the Linux Foundation that enables communication and interoperability between AI agents.

**What is A2A?**  

- Open protocol for agent-to-agent communication (Apache 2.0 License)

- Maintained by the Linux Foundation, contributed by Google

- Enables agents from different frameworks to collaborate securely

- Uses JSON-RPC 2.0 over HTTP(S) with standardized Agent Cards

In [ ]:
**SDK Installation:**
pip install a2a-sdk                    # Core SDK
pip install "a2a-sdk[http-server]"     # With HTTP server support
pip install "a2a-sdk[all]"             # All optional dependencies

**Architecture Components**: 

- **Agent Cards** - Self-describing agent manifests

- **Skills** - Agent capabilities with input/output specs

- **Tasks** - Stateful conversations between agents

- **Transport** - JSON-RPC, gRPC, HTTP+JSON

- **Discovery** - Capability-based agent discovery

- **Security** - OAuth, API keys, mTLS support

**This Guide Covers**:

1. Agent Card structure (official A2A types)

2. Agent Discovery & Skills

3. Task execution & messaging

4. Complete working examples with A2A SDK

## 1. Agent Card: Official A2A Structure

Agent Cards are self-describing manifests that provide:

- Identity (name, version, provider)

- Capabilities (skills, input/output modes)  

- Network endpoints (URL, transport protocols)

- Security requirements (OAuth, API keys, mTLS)

**Core Components from `a2a.types`**:

- `AgentCard` - Complete agent descriptor

- `AgentSkill` - Individual capability definition

- `AgentInterface` - Transport protocol + URL binding

- `AgentProvider` - Organization metadata

- `AgentCapabilities` - Optional protocol features

In [ ]:
"""
BEGINNER'S GUIDE: What is an Agent Card?
==========================================
Think of an Agent Card like a business card for AI agents. It tells other agents:
- Who am I? (name, version)
- What can I do? (skills like "data cleaning", "analysis")
- How to contact me? (URL endpoint)
- What data formats do I work with? (JSON, text, etc.)

This helps agents discover and communicate with each other automatically!
"""

import json
from typing import List, Dict, Any, Optional

# Example 1: Simple Agent Card - The Basics
# ==========================================
# This is the minimum information an agent needs to share

simple_agent_card = {
    # Identity - Who is this agent?
    "name": "Data Processor Agent",           # Human-readable name
    "version": "1.0.0",                       # Agent version
    "protocol_version": "0.3.0",               # A2A protocol version
    
    # Network - How to reach this agent
    "url": "https://api.example.com/agents/dataprocessor",  # Agent's address
    "preferred_transport": "JSONRPC",          # Communication method
    
    # Purpose - What does this agent do?
    "description": "Processes and cleans raw data for analytics pipelines",
    
    # Skills - What capabilities does this agent have?
    "skills": [
        {
            "id": "data-cleaning",                    # Unique skill identifier
            "name": "Data Cleaning",                  # Human-readable name
            "description": "Clean and normalize raw datasets",
            "tags": ["data", "preprocessing", "ETL"], # Keywords for discovery
            "examples": [                             # Example requests
                "Clean this customer database",
                "Remove duplicates from sales data"
            ]
        }
    ],
    
    # Data Formats - What types of data can this agent handle?
    "default_input_modes": ["text/plain", "application/json"],   # What it accepts
    "default_output_modes": ["application/json"],                # What it returns
    
    # Capabilities - Special features
    "capabilities": {
        "streaming": False,              # Can it stream data?
        "push_notifications": False      # Can it send notifications?
    }
}

print("=== Simple A2A Agent Card ===")
print("This shows the basic information every agent needs to share.\n")
print(json.dumps(simple_agent_card, indent=2))

In [ ]:
# Example 2: Complete Agent Card with Advanced Features
complete_agent_card = {
    "name": "Advanced ML Agent",
    "version": "2.1.0",
    "protocol_version": "0.3.0",
    
    # Network Configuration
    "url": "https://ml-agent.company.com/a2a/v1",
    "preferred_transport": "JSONRPC",
    "additional_interfaces": [
        {
            "url": "https://ml-agent.company.com/a2a/v1",
            "transport": "JSONRPC"
        },
        {
            "url": "grpc://ml-agent.company.com:50051",
            "transport": "GRPC"
        }
    ],
    
    "description": "Advanced machine learning agent for predictive analytics and model training",
    "documentation_url": "https://docs.company.com/ml-agent",
    "icon_url": "https://cdn.company.com/icons/ml-agent.png",
    
    # Provider Information
    "provider": {
        "organization": "ML Solutions Inc.",
        "url": "https://mlsolutions.com"
    },
    
    # Skills Definition
    "skills": [
        {
            "id": "sentiment-analysis",
            "name": "Sentiment Analysis",
            "description": "Analyze sentiment in text data using transformer models",
            "tags": ["nlp", "ml", "sentiment", "classification"],
            "examples": [
                "Analyze sentiment of these customer reviews",
                "What's the sentiment of this tweet?"
            ],
            "input_modes": ["text/plain", "application/json"],
            "output_modes": ["application/json"]
        },
        {
            "id": "model-training",
            "name": "ML Model Training",
            "description": "Train custom machine learning models on provided datasets",
            "tags": ["ml", "training", "deep-learning"],
            "examples": [
                "Train a classifier on this dataset",
                "Build a prediction model for sales forecasting"
            ],
            "input_modes": ["application/json"],
            "output_modes": ["application/json", "application/octet-stream"]
        },
        {
            "id": "prediction",
            "name": "Prediction Service",
            "description": "Generate predictions using trained models",
            "tags": ["ml", "inference", "prediction"],
            "examples": [
                "Predict customer churn for these users",
                "Forecast next quarter revenue"
            ]
        }
    ],
    
    # MIME Types
    "default_input_modes": ["text/plain", "application/json"],
    "default_output_modes": ["application/json"],
    
    # Capabilities
    "capabilities": {
        "streaming": True,
        "push_notifications": True,
        "persistent_context": True
    },
    
    # Security Configuration
    "security": [
        {"oauth": ["read", "write"]},  # Option 1: OAuth with scopes
        {"api-key": [], "mtls": []}     # Option 2: API key AND mTLS
    ],
    
    "security_schemes": {
        "oauth": {
            "type": "oauth2",
            "flows": {
                "authorizationCode": {
                    "authorization_url": "https://auth.company.com/oauth/authorize",
                    "token_url": "https://auth.company.com/oauth/token",
                    "scopes": {
                        "read": "Read access to agent data",
                        "write": "Write access to create tasks"
                    }
                }
            }
        },
        "api-key": {
            "type": "apiKey",
            "in": "header",
            "name": "X-API-Key"
        },
        "mtls": {
            "type": "mutualTLS"
        }
    },
    
    # Extended Card Support
    "supports_authenticated_extended_card": True
}

print("\n=== Complete A2A Agent Card ===")
print(json.dumps(complete_agent_card, indent=2)[:1500] + "\n... (truncated)")

# Key Features Summary
print("\n=== Key Features ===")
print(f"Skills: {len(complete_agent_card['skills'])}")
print(f"Transports: {[i['transport'] for i in complete_agent_card['additional_interfaces']]}")
print(f"Streaming: {complete_agent_card['capabilities']['streaming']}")
print(f"Security Options: {len(complete_agent_card['security'])} (OAuth OR API-key+mTLS)")

## 2. Agent Discovery: Finding the Right Agent

### What is the Directory Facilitator?

Imagine you need to find a plumber in your city. You could:

1. Search in the Yellow Pages by skill ("plumbing")

2. Filter by location (your area)

3. Check availability (are they working today?)

The **Directory Facilitator (DF)** is like the Yellow Pages for AI agents! It helps agents:

- **Register** themselves when they come online

- **Search** for other agents with specific skills

- **Subscribe** to events (when new agents join or leave)

### How It Works (Simple Steps):

In [ ]:
Step 1: Registration
   Agent → "I'm DataCleaner, I can clean data" → Directory

Step 2: Search
   Manager → "Who can clean data?" → Directory → "DataCleaner!"

Step 3: Communication
   Manager → "Clean this file" → DataCleaner

### Key Features:

- **Fast Indexing**: Finds agents quickly using skill and type indices

- **Event System**: Get notified when agents join/leave

- **Thread-Safe**: Multiple searches can happen at the same time

- **Statistics**: Track how many agents are active, suspended, etc.

In [ ]:
"""
IMPLEMENTATION: Directory Facilitator and Search
=================================================
This code creates a centralized registry where agents can:
1. Register themselves
2. Search for other agents
3. Subscribe to events
"""

from typing import Callable, Optional, Set
from collections import defaultdict
from dataclasses import dataclass, field
from enum import Enum
from datetime import datetime
import threading
import uuid

# First, let's define the basic data structures we need

class AgentState(Enum):
    """Agent lifecycle states - like a traffic light for agents"""
    INITIATED = "initiated"          # Just created, not ready yet
    REGISTERED = "registered"        # Signed up in the directory
    ACTIVE = "active"                # Working and accepting tasks
    SUSPENDED = "suspended"          # Temporarily paused
    MIGRATING = "migrating"          # Moving to another server
    TERMINATING = "terminating"      # Shutting down gracefully
    TERMINATED = "terminated"        # Completely stopped

@dataclass
class QoSParameters:
    """Quality of Service - How reliable is this agent?"""
    availability: float = 0.99        # Uptime: 0.99 = 99% available
    response_time_ms: int = 100       # How fast it responds
    throughput: int = 100             # Tasks per minute

@dataclass
class AgentCard:
    """Complete agent profile - like a LinkedIn profile for agents"""
    # Required fields
    agent_id: str                                    # Unique identifier
    name: str                                        # Human-readable name
    agent_type: str                                  # Category (e.g., "processor", "analyzer")
    skills: List[str] = field(default_factory=list) # What it can do
    
    # Optional fields
    state: AgentState = AgentState.INITIATED
    endpoints: Dict[str, str] = field(default_factory=dict)  # How to reach it
    tags: List[str] = field(default_factory=list)            # Keywords for search
    qos: QoSParameters = field(default_factory=QoSParameters)
    max_conversations: int = 10                               # How many tasks at once
    created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())
    updated_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())

class SearchCriteria:
    """
    Search filter - like using filters on an e-commerce website
    Example: Find all agents that are:
    - Type: "processor"
    - Have skill: "data_cleaning"
    - State: ACTIVE
    - Availability: > 95%
    """
    def __init__(self,
                 agent_type: Optional[str] = None,
                 skills: Optional[List[str]] = None,
                 state: Optional[AgentState] = None,
                 tags: Optional[List[str]] = None,
                 min_availability: Optional[float] = None):
        self.agent_type = agent_type
        self.skills = skills or []
        self.state = state
        self.tags = tags or []
        self.min_availability = min_availability
    
    def matches(self, card: AgentCard) -> bool:
        """Check if an agent matches all criteria (like applying filters)"""
        # Type match - must be exact
        if self.agent_type and card.agent_type != self.agent_type:
            return False
        
        # Skills match - agent must have ALL requested skills
        if self.skills and not all(skill in card.skills for skill in self.skills):
            return False
        
        # State match - must be exact
        if self.state and card.state != self.state:
            return False
        
        # Tags match - agent must have at least ONE requested tag
        if self.tags and not any(tag in card.tags for tag in self.tags):
            return False
        
        # Availability - must meet minimum requirement
        if self.min_availability and card.qos.availability < self.min_availability:
            return False
        
        return True  # Passed all filters!

class DirectoryFacilitator:
    """
    The Yellow Pages for agents - Central registry for discovery
    
    How it works:
    1. Agents register when they start
    2. Other agents search for specific skills
    3. Directory uses indices for fast lookup
    """
    
    def __init__(self):
        # Main storage: agent_id → AgentCard
        self._registry: Dict[str, AgentCard] = {}
        
        # Fast lookup indices (like book indices)
        self._skill_index: Dict[str, Set[str]] = defaultdict(set)  # skill → agent_ids
        self._type_index: Dict[str, Set[str]] = defaultdict(set)   # type → agent_ids
        
        # Event system for notifications
        self._subscribers: Dict[str, List[Callable]] = defaultdict(list)
        
        # Thread safety lock (prevents conflicts when multiple agents register at once)
        self._lock = threading.RLock()
    
    def register(self, card: AgentCard) -> bool:
        """
        Register a new agent in the directory
        Like adding a business to the Yellow Pages
        """
        with self._lock:  # Lock to prevent conflicts
            # Check if already registered
            if card.agent_id in self._registry:
                print(f"❌ Agent {card.agent_id} already registered!")
                return False
            
            # Add to main registry
            self._registry[card.agent_id] = card
            
            # Update state
            card.state = AgentState.REGISTERED
            card.updated_at = datetime.utcnow().isoformat()
            
            # Update indices for fast search
            for skill in card.skills:
                self._skill_index[skill].add(card.agent_id)
            
            self._type_index[card.agent_type].add(card.agent_id)
            
            # Notify subscribers (like sending a newsletter)
            self._notify("agent_registered", card)
            
            print(f"✅ Registered: {card.name} (ID: {card.agent_id})")
            return True
    
    def deregister(self, agent_id: str) -> bool:
        """Remove agent from directory (agent is shutting down)"""
        with self._lock:
            if agent_id not in self._registry:
                return False
            
            card = self._registry[agent_id]
            
            # Clean up indices
            for skill in card.skills:
                self._skill_index[skill].discard(agent_id)
            
            self._type_index[card.agent_type].discard(agent_id)
            
            # Update state before removal
            card.state = AgentState.TERMINATED
            self._notify("agent_deregistered", card)
            
            # Remove from registry
            del self._registry[agent_id]
            print(f"🗑️  Deregistered: {card.name}")
            return True
    
    def search(self, criteria: SearchCriteria) -> List[AgentCard]:
        """
        Search for agents matching criteria
        Uses indices for fast lookup (O(1) instead of O(n))
        """
        with self._lock:
            results = []
            
            # Start with all agents
            candidate_ids = set(self._registry.keys())
            
            # Optimize using indices when possible
            if criteria.skills:
                # Get agents that have ALL required skills (set intersection)
                skill_sets = [self._skill_index.get(skill, set()) for skill in criteria.skills]
                if skill_sets:
                    candidate_ids &= set.intersection(*skill_sets)
            
            if criteria.agent_type:
                # Filter by type
                candidate_ids &= self._type_index.get(criteria.agent_type, set())
            
            # Apply full criteria matching on candidates
            for agent_id in candidate_ids:
                card = self._registry[agent_id]
                if criteria.matches(card):
                    results.append(card)
            
            return results
    
    def get(self, agent_id: str) -> Optional[AgentCard]:
        """Get specific agent by ID"""
        return self._registry.get(agent_id)
    
    def subscribe(self, event: str, callback: Callable[[AgentCard], None]):
        """
        Subscribe to events (like subscribing to notifications)
        Events: "agent_registered", "agent_deregistered", "agent_state_changed"
        """
        self._subscribers[event].append(callback)
    
    def update(self, agent_id: str, updates: Dict[str, Any]) -> bool:
        """Update agent properties (e.g., change state to SUSPENDED)"""
        with self._lock:
            if agent_id not in self._registry:
                return False
            
            card = self._registry[agent_id]
            old_state = card.state
            
            # Apply updates
            for key, value in updates.items():
                if hasattr(card, key):
                    setattr(card, key, value)
            
            card.updated_at = datetime.utcnow().isoformat()
            
            # Notify if state changed
            if old_state != card.state:
                self._notify("agent_state_changed", card)
            
            return True
    
    def _notify(self, event: str, card: AgentCard):
        """Internal: Notify all subscribers of an event"""
        for callback in self._subscribers[event]:
            try:
                callback(card)
            except Exception as e:
                print(f"⚠️  Subscriber error: {e}")
    
    def get_statistics(self) -> Dict[str, Any]:
        """Get registry statistics (dashboard view)"""
        with self._lock:
            state_counts = defaultdict(int)
            for card in self._registry.values():
                state_counts[card.state.value] += 1
            
            return {
                "total_agents": len(self._registry),
                "by_state": dict(state_counts),
                "unique_skills": len(self._skill_index),
                "unique_types": len(self._type_index)
            }

print("✅ Directory Facilitator implementation complete!")
print("📚 You can now register agents, search for them, and track events.")

### Flow Diagrams: Understanding Directory Facilitator Operations

The Directory Facilitator uses **optimized indexing** and **event-driven architecture** for high-performance agent discovery.

---

#### **FLOW 1: Agent Registration**

In [ ]:
Agent                Directory Facilitator              Indices
  │                          │                            │
  │  register(AgentCard)     │                            │
  ├─────────────────────────>│                            │
  │                          │                            │
  │                          │ 1. Check if exists         │
  │                          │    (agent_id in registry?) │
  │                          │                            │
  │                          │ 2. Add to registry         │
  │                          │    _registry[id] = card    │
  │                          │                            │
  │                          │ 3. Set state = REGISTERED  │
  │                          │                            │
  │                          │ 4. Update skill index      │
  │                          ├───────────────────────────>│
  │                          │  for skill in skills:      │
  │                          │    _skill_index[skill]     │
  │                          │      .add(agent_id)        │
  │                          │                            │
  │                          │ 5. Update type index       │
  │                          ├───────────────────────────>│
  │                          │  _type_index[type]         │
  │                          │    .add(agent_id)          │
  │                          │                            │
  │                          │ 6. Notify subscribers      │
  │                          │    _notify("agent_         │
  │                          │      registered", card)    │
  │                          │                            │
  │  ✓ Success               │                            │
  │<─────────────────────────┤                            │

**Result**: Agent is now discoverable via skill/type queries.

---

#### **FLOW 2: Agent Search (Index-Optimized)**

In [ ]:
Client               Directory Facilitator          Indices
  │                          │                         │
  │  search(criteria)        │                         │
  ├─────────────────────────>│                         │
  │  skills: ["validation"]  │                         │
  │  type: "processor"       │                         │
  │                          │                         │
  │                          │ 1. Get skill matches    │
  │                          │<────────────────────────┤
  │                          │  _skill_index           │
  │                          │    ["validation"]       │
  │                          │  → {001, 003}           │
  │                          │                         │
  │                          │ 2. Intersect all skills │
  │                          │    candidates =         │
  │                          │      skill_sets ∩       │
  │                          │                         │
  │                          │ 3. Get type matches     │
  │                          │<────────────────────────┤
  │                          │  _type_index            │
  │                          │    ["processor"]        │
  │                          │  → {001, 003}           │
  │                          │                         │
  │                          │ 4. Intersect with type  │
  │                          │    candidates &=        │
  │                          │      type_set           │
  │                          │                         │
  │                          │ 5. Apply full criteria  │
  │                          │    (state, tags, QoS)   │
  │                          │    Filter candidates    │
  │                          │                         │
  │  [results]               │                         │
  │<─────────────────────────┤                         │

**Optimization**: Index lookup O(1) vs full scan O(n). With 1000 agents: ~6 operations instead of 10,000.

---

#### **FLOW 3: Event Subscription & Notification**

In [ ]:
Subscriber           Directory Facilitator        Event System
    │                        │                         │
    │  subscribe(event, cb)  │                         │
    ├───────────────────────>│                         │
    │                        │ Store callback          │
    │                        ├────────────────────────>│
    │                        │ _subscribers[event]     │
    │                        │   .append(callback)     │
    │  ✓ Subscribed          │                         │
    │<───────────────────────┤                         │
   ...                      ...                       ...
    │                        │                         │
    │  [REGISTRATION EVENT]  │                         │
    │                        │                         │
    │                        │ _notify(event, card)    │
    │                        ├────────────────────────>│
    │                        │ for cb in subscribers:  │
    │                        │   cb(card)              │
    │                        │                         │
    │  callback(card)        │                         │
    │<───────────────────────┤                         │
    │  [Process event]       │                         │

**Events**: `agent_registered`, `agent_deregistered`, `agent_state_changed`

---

#### **FLOW 4: Agent Deregistration (Cleanup)**

In [ ]:
Agent                Directory Facilitator          Indices
  │                          │                         │
  │  deregister(agent_id)    │                         │
  ├─────────────────────────>│                         │
  │                          │                         │
  │                          │ 1. Get agent card       │
  │                          │                         │
  │                          │ 2. Clean skill index    │
  │                          ├────────────────────────>│
  │                          │  for skill:             │
  │                          │    _skill_index[skill]  │
  │                          │      .discard(id)       │
  │                          │                         │
  │                          │ 3. Clean type index     │
  │                          ├────────────────────────>│
  │                          │  _type_index[type]      │
  │                          │    .discard(id)         │
  │                          │                         │
  │                          │ 4. Set TERMINATED state │
  │                          │                         │
  │                          │ 5. Notify subscribers   │
  │                          │                         │
  │                          │ 6. Remove from registry │
  │                          │    del _registry[id]    │
  │                          │                         │
  │  ✓ Deregistered          │                         │
  │<─────────────────────────┤                         │

**Result**: Agent removed, all indices cleaned, subscribers notified.

---

#### **Data Structures: Internal Organization**

In [ ]:
**1. Main Registry** (Hash Map)
_registry: Dict[agent_id → AgentCard]
{
  "agent-001": AgentCard(skills=["cleaning"], ...),
  "agent-002": AgentCard(skills=["analysis"], ...)
}

In [ ]:
**2. Skill Index** (Inverted Index)
_skill_index: Dict[skill → Set[agent_id]]
{
  "cleaning":    {"agent-001", "agent-003"},
  "validation":  {"agent-003"}
}
# Enables O(1): "Which agents have skill X?"

In [ ]:
**3. Type Index** (Inverted Index)
_type_index: Dict[agent_type → Set[agent_id]]
{
  "processor": {"agent-001", "agent-003"},
  "analyzer":  {"agent-002"}
}

In [ ]:
**4. Subscribers** (Pub-Sub)
_subscribers: Dict[event → List[Callable]]
{
  "agent_registered": [callback1, callback2],
  "agent_state_changed": [callback3]
}

**5. Thread Lock** (RLock) - Ensures thread-safe concurrent operations

---

#### **Concurrency Model**

All mutating operations acquire `_lock` (RLock):

In [ ]:
THREAD 1                      THREAD 2
register(agent_A)             search(criteria)
    │                             │
    ├─> acquire(_lock)            ├─> acquire(_lock)
    │   [LOCKED]                  │   [WAITING...]
    │                             │
    ├─> update registry           │
    ├─> update indices            │
    ├─> notify subscribers        │
    │                             │
    └─> release(_lock)            │
        [UNLOCKED]                └─> [ACQUIRED]
                                      read data
                                      release(_lock)

✓ Prevents race conditions  

✓ Ensures consistency (indices ↔ registry)  

✓ Reentrant: Same thread can acquire multiple times

In [ ]:
"""
HANDS-ON EXAMPLE: Using the Directory Facilitator
===================================================
Let's see how agents register, search, and communicate!
"""

# Step 1: Create the Directory
# ==============================
df = DirectoryFacilitator()

# Step 2: Set up Event Listeners (Optional but helpful!)
# ======================================================
# These functions get called when events happen

def on_agent_registered(card: AgentCard):
    """Called when a new agent registers"""
    print(f"📢 [EVENT] New agent joined: {card.name} ({card.agent_id})")
    print(f"   Skills: {', '.join(card.skills)}")

def on_state_changed(card: AgentCard):
    """Called when an agent's state changes"""
    print(f"📢 [EVENT] {card.name} changed state → {card.state.value}")

# Subscribe to events
df.subscribe("agent_registered", on_agent_registered)
df.subscribe("agent_state_changed", on_state_changed)

print("="*60)
print("DIRECTORY FACILITATOR DEMO")
print("="*60)

# Step 3: Create and Register Agents
# ===================================
print("\n📝 STEP 1: Registering Agents")
print("-" * 60)

agents = [
    AgentCard(
        agent_id="agent-001",
        name="DataCleaner",
        agent_type="processor",
        skills=["data_cleaning", "validation"],
        tags=["data", "production"],
        qos=QoSParameters(availability=0.95)
    ),
    AgentCard(
        agent_id="agent-002",
        name="MLModel",
        agent_type="ai-analyzer",
        skills=["prediction", "classification"],
        tags=["ml", "critical"],
        qos=QoSParameters(availability=0.999)
    ),
    AgentCard(
        agent_id="agent-003",
        name="DataValidator",
        agent_type="processor",
        skills=["validation", "schema_check"],
        tags=["data", "quality"],
        qos=QoSParameters(availability=0.97)
    )
]

# Register each agent
for agent in agents:
    df.register(agent)

# Step 4: Search for Agents
# ==========================
print("\n" + "="*60)
print("🔍 STEP 2: Searching for Agents")
print("="*60)

# Search 1: Find all processors
print("\n1️⃣ Find all 'processor' type agents:")
print("-" * 40)
results = df.search(SearchCriteria(agent_type="processor"))
for card in results:
    print(f"   ✓ {card.name}")
    print(f"     Skills: {', '.join(card.skills)}")
    print(f"     Availability: {card.qos.availability * 100}%")

# Search 2: Find agents with specific skill
print("\n2️⃣ Find agents with 'validation' skill:")
print("-" * 40)
results = df.search(SearchCriteria(skills=["validation"]))
for card in results:
    print(f"   ✓ {card.name}: {', '.join(card.skills)}")

# Search 3: Find high-availability ML agents
print("\n3️⃣ Find ML agents with >98% availability:")
print("-" * 40)
results = df.search(SearchCriteria(tags=["ml"], min_availability=0.98))
for card in results:
    print(f"   ✓ {card.name}")
    print(f"     Availability: {card.qos.availability * 100}%")
    print(f"     Tags: {', '.join(card.tags)}")

# Step 5: Update Agent State
# ===========================
print("\n" + "="*60)
print("🔄 STEP 3: Updating Agent State")
print("="*60)
print("\nSuspending MLModel for maintenance...")
df.update("agent-002", {"state": AgentState.SUSPENDED})

# Step 6: View Statistics
# ========================
print("\n" + "="*60)
print("📊 STEP 4: Directory Statistics")
print("="*60)
stats = df.get_statistics()
print(f"\n📈 Total Agents: {stats['total_agents']}")
print(f"📈 Unique Skills: {stats['unique_skills']}")
print(f"📈 Unique Types: {stats['unique_types']}")
print(f"\n🚦 Agents by State:")
for state, count in stats['by_state'].items():
    print(f"   {state}: {count}")

print("\n" + "="*60)
print("✅ DEMO COMPLETE!")
print("="*60)

## 3. Agent Skills: What Can Agents Do?

### Understanding Skills

Think of skills like job requirements on LinkedIn:

- **Skill Name**: "Data Cleaning"

- **Description**: What the skill does

- **Inputs**: What data it needs (like function parameters)

- **Outputs**: What it produces (like return values)

- **Keywords**: Tags for easy discovery

### Why Skills Matter

Skills help agents:

1. **Advertise** what they can do

2. **Discover** other agents with complementary skills

3. **Compose** workflows (chain skills together)

4. **Validate** if they can handle a request

### Example Skill Chain

In [ ]:
Raw Data → [Data Cleaning] → Clean Data → [Feature Engineering] → Features → [Model Training] → Trained Model

Each arrow represents data flowing from one skill's output to the next skill's input!

In [ ]:
"""
IMPLEMENTATION: Skill Registry
================================
Skills define what agents can do - like a menu at a restaurant!
"""

from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Set
from collections import defaultdict

@dataclass
class SkillParameter:
    """
    A single input or output parameter
    Like a function parameter: name, type, required/optional, default value
    """
    name: str                    # Parameter name (e.g., "dataset")
    param_type: str              # Data type ("string", "int", "float", "bool", "object", "array")
    required: bool = True        # Is this parameter mandatory?
    description: str = ""        # What does this parameter do?
    default: Any = None          # Default value if not provided
    constraints: Dict[str, Any] = field(default_factory=dict)  # Validation rules (min, max, pattern)

@dataclass
class Skill:
    """
    Complete skill definition - what an agent can do
    Think of this as a function signature with metadata
    """
    skill_id: str                 # Unique identifier (e.g., "data_cleaning")
    name: str                     # Human-readable name
    description: str              # What does this skill do?
    
    # Interface (inputs and outputs)
    inputs: List[SkillParameter] = field(default_factory=list)   # What it needs
    outputs: List[SkillParameter] = field(default_factory=list)  # What it produces
    
    # Semantic (for discovery)
    category: str = "general"                           # Skill category
    keywords: List[str] = field(default_factory=list)   # Search tags
    
    # Execution properties
    preconditions: List[str] = field(default_factory=list)   # What must be true before execution
    postconditions: List[str] = field(default_factory=list)  # What will be true after execution
    
    # Quality of Service
    avg_execution_time_ms: int = 1000   # Average time to complete
    deterministic: bool = True           # Same input → same output?
    idempotent: bool = False            # Can run multiple times safely?
    
    # Composition (skill chaining)
    composable_with: List[str] = field(default_factory=list)  # Compatible skill IDs
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for JSON serialization"""
        return asdict(self)

class SkillRegistry:
    """
    Central registry for all skills - like a phonebook for capabilities
    
    Features:
    - Register skill definitions
    - Search by category or keywords
    - Find compatible skills for composition
    """
    
    def __init__(self):
        self._skills: Dict[str, Skill] = {}                      # Main storage
        self._category_index: Dict[str, Set[str]] = defaultdict(set)  # category → skill_ids
        self._keyword_index: Dict[str, Set[str]] = defaultdict(set)   # keyword → skill_ids
    
    def register(self, skill: Skill) -> bool:
        """
        Register a new skill
        Returns: True if successful, False if already exists
        """
        if skill.skill_id in self._skills:
            print(f"❌ Skill '{skill.skill_id}' already registered!")
            return False
        
        # Add to main registry
        self._skills[skill.skill_id] = skill
        
        # Update indices for fast search
        self._category_index[skill.category].add(skill.skill_id)
        
        for keyword in skill.keywords:
            self._keyword_index[keyword.lower()].add(skill.skill_id)
        
        print(f"✅ Registered skill: {skill.name} (ID: {skill.skill_id})")
        return True
    
    def get(self, skill_id: str) -> Optional[Skill]:
        """Get skill by ID"""
        return self._skills.get(skill_id)
    
    def search_by_category(self, category: str) -> List[Skill]:
        """
        Find all skills in a category
        Example: category="data-processing"
        """
        skill_ids = self._category_index.get(category, set())
        return [self._skills[sid] for sid in skill_ids]
    
    def search_by_keywords(self, keywords: List[str]) -> List[Skill]:
        """
        Find skills matching any of the keywords (OR logic)
        Example: keywords=["ml", "prediction"]
        """
        skill_ids = set()
        for keyword in keywords:
            skill_ids.update(self._keyword_index.get(keyword.lower(), set()))
        return [self._skills[sid] for sid in skill_ids]
    
    def find_compatible_inputs(self, outputs: List[SkillParameter]) -> List[Skill]:
        """
        Find skills that can use these outputs as inputs
        Useful for building skill chains!
        
        Example: If skill A outputs "cleaned_data", 
        find all skills that accept "cleaned_data" as input
        """
        compatible = []
        output_types = {out.name: out.param_type for out in outputs}
        
        for skill in self._skills.values():
            # Check if skill inputs match output types
            matches = True
            for inp in skill.inputs:
                if inp.required and inp.name not in output_types:
                    matches = False
                    break
                if inp.name in output_types and output_types[inp.name] != inp.param_type:
                    matches = False
                    break
            
            if matches:
                compatible.append(skill)
        
        return compatible
    
    def compose_chain(self, skill_ids: List[str]) -> bool:
        """
        Validate if skills can be composed in sequence
        Returns: True if valid chain, False otherwise
        
        Example: Can we chain [data_cleaning → feature_engineering → model_training]?
        """
        for i in range(len(skill_ids) - 1):
            current = self._skills.get(skill_ids[i])
            next_skill = self._skills.get(skill_ids[i + 1])
            
            if not current or not next_skill:
                return False
            
            # Check output → input compatibility
            compatible = self.find_compatible_inputs(current.outputs)
            if next_skill not in compatible:
                print(f"❌ Incompatible: {current.name} → {next_skill.name}")
                return False
        
        print(f"✅ Valid skill chain!")
        return True

print("✅ Skill Registry implementation complete!")
print("📚 You can now define skills, register them, and build skill chains.")

In [ ]:
"""
HANDS-ON EXAMPLE: Working with Skills
=======================================
Let's create a data processing pipeline with skills!
"""

# Create skill registry
skill_registry = SkillRegistry()

print("="*60)
print("SKILL REGISTRY DEMO")
print("="*60)

# Step 1: Define Skills
# ======================
print("\n📝 STEP 1: Defining Skills")
print("-" * 60)

# Skill 1: Data Cleaning
data_cleaning = Skill(
    skill_id="data_cleaning",
    name="Data Cleaning",
    description="Clean and preprocess raw data - remove duplicates, fill nulls, fix formats",
    
    # What this skill needs as input
    inputs=[
        SkillParameter("raw_data", "object", description="Raw dataset to clean"),
        SkillParameter("operations", "array", required=False, 
                      default=["remove_duplicates", "fill_nulls"],
                      description="List of cleaning operations to perform")
    ],
    
    # What this skill produces as output
    outputs=[
        SkillParameter("cleaned_data", "object", description="Cleaned dataset ready for analysis"),
        SkillParameter("cleaning_report", "object", description="Statistics about cleaning (rows removed, nulls filled, etc.)")
    ],
    
    # Metadata for discovery
    category="data-processing",
    keywords=["cleaning", "preprocessing", "data", "etl", "validation"],
    
    # Performance characteristics
    avg_execution_time_ms=500,
    deterministic=True,    # Same input always gives same output
    idempotent=True,       # Safe to run multiple times
    composable_with=["feature_engineering", "validation"]
)

# Skill 2: Feature Engineering
feature_engineering = Skill(
    skill_id="feature_engineering",
    name="Feature Engineering",
    description="Create new features from cleaned data for machine learning",
    
    inputs=[
        SkillParameter("cleaned_data", "object", description="Cleaned dataset"),
        SkillParameter("feature_specs", "array", description="Specifications for features to create")
    ],
    
    outputs=[
        SkillParameter("feature_matrix", "object", description="Matrix of engineered features"),
        SkillParameter("feature_names", "array", description="List of feature names created")
    ],
    
    category="data-processing",
    keywords=["features", "engineering", "ml", "transformation", "preprocessing"],
    preconditions=["cleaned_data.is_valid"],  # Data must be valid
    avg_execution_time_ms=800
)

# Skill 3: Model Training
model_training = Skill(
    skill_id="model_training",
    name="Model Training",
    description="Train machine learning model on prepared features",
    
    inputs=[
        SkillParameter("feature_matrix", "object", description="Training features"),
        SkillParameter("labels", "array", description="Target labels for supervised learning"),
        SkillParameter("algorithm", "string", required=False, default="random_forest",
                      description="ML algorithm to use")
    ],
    
    outputs=[
        SkillParameter("trained_model", "object", description="Trained ML model ready for predictions"),
        SkillParameter("metrics", "object", description="Training metrics (accuracy, loss, etc.)")
    ],
    
    category="machine-learning",
    keywords=["training", "ml", "model", "ai", "prediction"],
    preconditions=["feature_matrix.shape[0] > 0"],     # Must have data
    postconditions=["trained_model.is_fitted"],        # Model must be trained
    avg_execution_time_ms=5000,
    deterministic=False  # May vary due to random initialization
)

# Step 2: Register Skills
# ========================
print("\nRegistering skills in the registry...")
for skill in [data_cleaning, feature_engineering, model_training]:
    skill_registry.register(skill)

# Step 3: Search Skills
# ======================
print("\n" + "="*60)
print("🔍 STEP 2: Searching for Skills")
print("="*60)

# Search by category
print("\n1️⃣ All skills in 'data-processing' category:")
print("-" * 40)
results = skill_registry.search_by_category("data-processing")
for skill in results:
    print(f"   ✓ {skill.name}")
    print(f"     Input: {', '.join([p.name for p in skill.inputs])}")
    print(f"     Output: {', '.join([p.name for p in skill.outputs])}")

# Search by keywords
print("\n2️⃣ Skills matching 'ml' keyword:")
print("-" * 40)
results = skill_registry.search_by_keywords(["ml"])
for skill in results:
    print(f"   ✓ {skill.name}")
    print(f"     Keywords: {', '.join(skill.keywords)}")

# Step 4: Test Skill Composition
# ===============================
print("\n" + "="*60)
print("🔗 STEP 3: Testing Skill Composition")
print("="*60)

# Can we build a pipeline?
print("\n1️⃣ Finding skills compatible with 'data_cleaning' output:")
print("-" * 40)
compatible = skill_registry.find_compatible_inputs(data_cleaning.outputs)
for skill in compatible:
    print(f"   ✓ {skill.name} - Can accept cleaned_data")

# Validate complete chain
print("\n2️⃣ Validating skill chain:")
print("-" * 40)
chain = ["data_cleaning", "feature_engineering", "model_training"]
print(f"   Chain: {' → '.join(chain)}")
is_valid = skill_registry.compose_chain(chain)

if is_valid:
    print(f"\n   ✅ This forms a valid ML pipeline!")
    print(f"   Flow: Raw Data → Cleaned Data → Features → Trained Model")
else:
    print(f"\n   ❌ Invalid chain - outputs don't match inputs")

print("\n" + "="*60)
print("✅ DEMO COMPLETE!")
print("="*60)
print("\nKey Takeaway: Skills define what agents can do,")
print("and the registry helps discover and compose them!")

## 4. Agent Lifecycle: Complete State Machine

Agents transition through well-defined states: Initiated → Registered → Active → Suspended → Terminating → Terminated (with Migrating for relocation).

In [ ]:
from typing import Callable, Dict
from datetime import datetime, timedelta
import time

class LifecycleEvent:
    """Lifecycle event data"""
    def __init__(self, event_type: str, agent_id: str, from_state: AgentState, 
                 to_state: AgentState, metadata: Dict[str, Any] = None):
        self.event_type = event_type
        self.agent_id = agent_id
        self.from_state = from_state
        self.to_state = to_state
        self.timestamp = datetime.utcnow().isoformat()
        self.metadata = metadata or {}

class LifecycleManager:
    """Manages agent lifecycle state transitions with validation"""
    
    # Valid state transitions
    TRANSITIONS = {
        AgentState.INITIATED: [AgentState.REGISTERED, AgentState.TERMINATED],
        AgentState.REGISTERED: [AgentState.ACTIVE, AgentState.TERMINATED],
        AgentState.ACTIVE: [AgentState.SUSPENDED, AgentState.MIGRATING, AgentState.TERMINATING],
        AgentState.SUSPENDED: [AgentState.ACTIVE, AgentState.TERMINATING],
        AgentState.MIGRATING: [AgentState.ACTIVE, AgentState.TERMINATING],
        AgentState.TERMINATING: [AgentState.TERMINATED],
        AgentState.TERMINATED: []  # Final state
    }
    
    def __init__(self, directory: DirectoryFacilitator):
        self.directory = directory
        self._event_handlers: Dict[str, List[Callable]] = defaultdict(list)
        self._heartbeat_tracking: Dict[str, datetime] = {}
        self._recovery_attempts: Dict[str, int] = defaultdict(int)
    
    def transition(self, agent_id: str, to_state: AgentState, 
                  metadata: Dict[str, Any] = None) -> bool:
        """Transition agent to new state with validation"""
        card = self.directory.get(agent_id)
        if not card:
            print(f"Agent {agent_id} not found")
            return False
        
        from_state = card.state
        
        # Validate transition
        if to_state not in self.TRANSITIONS.get(from_state, []):
            print(f"Invalid transition: {from_state.value} -> {to_state.value}")
            return False
        
        # Pre-transition hook
        if not self._on_before_transition(card, from_state, to_state):
            return False
        
        # Update state
        self.directory.update(agent_id, {"state": to_state})
        
        # Create event
        event = LifecycleEvent("state_transition", agent_id, from_state, to_state, metadata)
        
        # Post-transition hook
        self._on_after_transition(card, event)
        
        # Notify event handlers
        self._notify_handlers("transition", event)
        
        print(f"[LIFECYCLE] {card.name}: {from_state.value} -> {to_state.value}")
        return True
    
    def bootstrap(self, card: AgentCard) -> bool:
        """Bootstrap new agent through initialization"""
        # Validate configuration
        if not self._validate_configuration(card):
            return False
        
        # Register with directory
        if not self.directory.register(card):
            return False
        
        # Start heartbeat tracking
        self._heartbeat_tracking[card.agent_id] = datetime.utcnow()
        
        print(f"[BOOTSTRAP] {card.name} initialized")
        return True
    
    def activate(self, agent_id: str) -> bool:
        """Activate registered agent"""
        return self.transition(agent_id, AgentState.ACTIVE)
    
    def suspend(self, agent_id: str, reason: str = "") -> bool:
        """Suspend active agent"""
        return self.transition(agent_id, AgentState.SUSPENDED, {"reason": reason})
    
    def resume(self, agent_id: str) -> bool:
        """Resume suspended agent"""
        card = self.directory.get(agent_id)
        if card and card.state == AgentState.SUSPENDED:
            return self.transition(agent_id, AgentState.ACTIVE)
        return False
    
    def shutdown(self, agent_id: str, graceful: bool = True) -> bool:
        """Shutdown agent"""
        card = self.directory.get(agent_id)
        if not card:
            return False
        
        if graceful:
            # Two-phase shutdown
            if card.state != AgentState.TERMINATING:
                self.transition(agent_id, AgentState.TERMINATING)
            
            # Cleanup resources
            self._cleanup_resources(agent_id)
        
        # Final termination
        self.transition(agent_id, AgentState.TERMINATED)
        
        # Deregister
        self.directory.deregister(agent_id)
        
        # Stop tracking
        self._heartbeat_tracking.pop(agent_id, None)
        self._recovery_attempts.pop(agent_id, None)
        
        return True
    
    def heartbeat(self, agent_id: str) -> bool:
        """Update agent heartbeat"""
        if agent_id in self._heartbeat_tracking:
            self._heartbeat_tracking[agent_id] = datetime.utcnow()
            return True
        return False
    
    def check_health(self, agent_id: str, timeout_seconds: int = 30) -> bool:
        """Check if agent is healthy based on heartbeat"""
        if agent_id not in self._heartbeat_tracking:
            return False
        
        last_heartbeat = self._heartbeat_tracking[agent_id]
        elapsed = (datetime.utcnow() - last_heartbeat).total_seconds()
        
        return elapsed < timeout_seconds
    
    def recover(self, agent_id: str, max_attempts: int = 3) -> bool:
        """Attempt to recover failed agent"""
        if self._recovery_attempts[agent_id] >= max_attempts:
            print(f"[RECOVERY] Max attempts reached for {agent_id}")
            return False
        
        self._recovery_attempts[agent_id] += 1
        
        card = self.directory.get(agent_id)
        if not card:
            return False
        
        # Try to reactivate
        if card.state == AgentState.SUSPENDED:
            if self.resume(agent_id):
                print(f"[RECOVERY] Successfully recovered {card.name}")
                self._recovery_attempts[agent_id] = 0  # Reset counter
                return True
        
        return False
    
    def on_event(self, event_type: str, handler: Callable[[LifecycleEvent], None]):
        """Register event handler"""
        self._event_handlers[event_type].append(handler)
    
    def _validate_configuration(self, card: AgentCard) -> bool:
        """Validate agent configuration"""
        if not card.agent_id or not card.name:
            return False
        if not card.skills:
            print(f"Warning: Agent {card.name} has no skills defined")
        return True
    
    def _on_before_transition(self, card: AgentCard, from_state: AgentState, 
                             to_state: AgentState) -> bool:
        """Pre-transition validation hook"""
        # Custom validation logic
        if to_state == AgentState.ACTIVE and not card.endpoints:
            print(f"Cannot activate {card.name}: No endpoints configured")
            return False
        return True
    
    def _on_after_transition(self, card: AgentCard, event: LifecycleEvent):
        """Post-transition hook"""
        # Update timestamps, log events, etc.
        if event.to_state == AgentState.ACTIVE:
            self._heartbeat_tracking[card.agent_id] = datetime.utcnow()
    
    def _cleanup_resources(self, agent_id: str):
        """Cleanup agent resources before termination"""
        # Close connections, release locks, etc.
        print(f"[CLEANUP] Releasing resources for {agent_id}")
    
    def _notify_handlers(self, event_type: str, event: LifecycleEvent):
        """Notify registered event handlers"""
        for handler in self._event_handlers.get(event_type, []):
            try:
                handler(event)
            except Exception as e:
                print(f"Handler error: {e}")

print("Lifecycle Manager implementation complete")

    Lifecycle Manager implementation complete

In [ ]:
# Example: Complete lifecycle management
df = DirectoryFacilitator()
lifecycle = LifecycleManager(df)

# Register event handler
def on_transition(event: LifecycleEvent):
    print(f"  [EVENT] {event.agent_id}: {event.from_state.value} → {event.to_state.value}")

lifecycle.on_event("transition", on_transition)

# Create agent
agent = AgentCard(
    agent_id="agent-lc-001",
    name="TestAgent",
    agent_type="processor",
    skills=["test_skill"],
    endpoints={"http": "http://localhost:8000"}
)

print("=== Agent Lifecycle Demo ===")
print("\n1. Bootstrap")
lifecycle.bootstrap(agent)

print("\n2. Activate")
lifecycle.activate(agent.agent_id)

print("\n3. Heartbeat")
lifecycle.heartbeat(agent.agent_id)
is_healthy = lifecycle.check_health(agent.agent_id)
print(f"  Health check: {'✓ Healthy' if is_healthy else '✗ Unhealthy'}")

print("\n4. Suspend")
lifecycle.suspend(agent.agent_id, reason="Maintenance")

print("\n5. Resume")
lifecycle.resume(agent.agent_id)

print("\n6. Graceful Shutdown")
lifecycle.shutdown(agent.agent_id, graceful=True)

# Verify final state
final_card = df.get(agent.agent_id)
if not final_card:
    print("\n✓ Agent successfully deregistered")

    === Agent Lifecycle Demo ===

    1. Bootstrap

    [BOOTSTRAP] TestAgent initialized

    2. Activate

      [EVENT] agent-lc-001: registered → active

    [LIFECYCLE] TestAgent: registered -> active

    3. Heartbeat

      Health check: ✓ Healthy

    4. Suspend

      [EVENT] agent-lc-001: active → suspended

    [LIFECYCLE] TestAgent: active -> suspended

    5. Resume

      [EVENT] agent-lc-001: suspended → active

    [LIFECYCLE] TestAgent: suspended -> active

    6. Graceful Shutdown

      [EVENT] agent-lc-001: active → terminating

    [LIFECYCLE] TestAgent: active -> terminating

    [CLEANUP] Releasing resources for agent-lc-001

      [EVENT] agent-lc-001: terminating → terminated

    [LIFECYCLE] TestAgent: terminating -> terminated

    ✓ Agent successfully deregistered

## 5. Message Protocol: How Agents Talk to Each Other

### What is ACL (Agent Communication Language)?

Think of ACL like email for agents:

- **Performative** = Subject line type (Request, Inform, Propose, etc.)

- **Sender/Receiver** = From/To addresses

- **Content** = The actual message body

- **Conversation ID** = Email thread ID (groups related messages)

### Common Performatives (Message Types)

| Performative | Meaning | Example |

|--------------|---------|---------|

| **REQUEST** | "Please do this" | "Please clean this dataset" |

| **INFORM** | "Here's information" | "The data is now clean" |

| **QUERY** | "What is...?" | "What is the status?" |

| **AGREE** | "I'll do it" | "Yes, I'll clean the data" |

| **REFUSE** | "I can't do it" | "Sorry, I'm at capacity" |

| **PROPOSE** | "How about this?" | "I can do it for $50" |

### Message Flow Example

In [ ]:
Agent A                    Agent B
   |                          |
   |  REQUEST: "Clean data"   |
   |------------------------->|
   |                          |
   |    AGREE: "Will do"      |
   |<-------------------------|
   |                          |
   |  INFORM_RESULT: "Done"   |
   |<-------------------------|

### Key Features

- **Conversation Tracking**: Groups messages into threads

- **Reply Management**: Links responses to original messages

- **Deadlines**: Set reply-by times

- **Content Negotiation**: Specify data formats (JSON, XML, etc.)

In [ ]:
class Performative(Enum):
    """FIPA ACL performatives with semantics"""
    # Information
    INFORM = "inform"          # Inform that a given proposition is true
    QUERY_IF = "query-if"      # Query whether a given proposition is true
    QUERY_REF = "query-ref"    # Query for objects matching a descriptor
    
    # Action
    REQUEST = "request"        # Request an action
    REQUEST_WHEN = "request-when"  # Request action when condition is true
    REQUEST_WHENEVER = "request-whenever"  # Request action whenever condition holds
    
    # Negotiation
    CFP = "cfp"               # Call for proposals
    PROPOSE = "propose"        # Submit a proposal
    ACCEPT_PROPOSAL = "accept-proposal"
    REJECT_PROPOSAL = "reject-proposal"
    
    # Confirmation
    AGREE = "agree"           # Agree to perform action
    REFUSE = "refuse"         # Refuse to perform action
    CONFIRM = "confirm"       # Confirm truth of proposition
    DISCONFIRM = "disconfirm" # Disconfirm proposition
    
    # Result
    INFORM_DONE = "inform-done"  # Inform action completed
    INFORM_RESULT = "inform-result"  # Inform result of action
    FAILURE = "failure"       # Inform action failed
    
    # Other
    NOT_UNDERSTOOD = "not-understood"
    PROPAGATE = "propagate"   # Forward to other agents

@dataclass
class ACLMessage:
    """FIPA ACL Message structure"""
    # Core
    performative: Performative
    sender: str
    receiver: str  # Can be extended to List[str] for multi-cast
    
    # Content
    content: Any
    language: str = "JSON"  # or "FIPA-SL", "XML", etc.
    ontology: str = "default"
    
    # Conversation
    protocol: str = "fipa-request"
    conversation_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    reply_with: str = field(default_factory=lambda: str(uuid.uuid4()))
    in_reply_to: Optional[str] = None
    reply_by: Optional[str] = None  # Deadline for reply
    
    # Message ID
    message_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    
    # Encoding
    encoding: str = "UTF-8"
    
    # Metadata
    timestamp: str = field(default_factory=lambda: datetime.utcnow().isoformat())
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "message_id": self.message_id,
            "performative": self.performative.value,
            "sender": self.sender,
            "receiver": self.receiver,
            "content": self.content,
            "language": self.language,
            "ontology": self.ontology,
            "protocol": self.protocol,
            "conversation_id": self.conversation_id,
            "reply_with": self.reply_with,
            "in_reply_to": self.in_reply_to,
            "reply_by": self.reply_by,
            "encoding": self.encoding,
            "timestamp": self.timestamp
        }
    
    def to_json(self) -> str:
        return json.dumps(self.to_dict(), indent=2)

class ConversationTracker:
    """Track conversation state and message threads"""
    
    def __init__(self):
        self._conversations: Dict[str, List[ACLMessage]] = defaultdict(list)
        self._reply_mapping: Dict[str, ACLMessage] = {}  # reply_with -> original message
    
    def add_message(self, msg: ACLMessage):
        """Add message to conversation"""
        self._conversations[msg.conversation_id].append(msg)
        self._reply_mapping[msg.reply_with] = msg
    
    def get_conversation(self, conversation_id: str) -> List[ACLMessage]:
        """Get all messages in conversation"""
        return self._conversations.get(conversation_id, [])
    
    def get_thread(self, message: ACLMessage) -> List[ACLMessage]:
        """Get message thread (original + replies)"""
        thread = [message]
        
        # Follow reply chain backwards
        current = message
        while current.in_reply_to:
            parent = self._reply_mapping.get(current.in_reply_to)
            if parent:
                thread.insert(0, parent)
                current = parent
            else:
                break
        
        return thread
    
    def get_pending_replies(self, agent_id: str) -> List[ACLMessage]:
        """Get messages awaiting reply from agent"""
        pending = []
        for conv_msgs in self._conversations.values():
            for msg in conv_msgs:
                if msg.receiver == agent_id and msg.reply_by:
                    # Check if already replied
                    replied = any(m.in_reply_to == msg.reply_with for m in conv_msgs)
                    if not replied:
                        pending.append(msg)
        return pending

print("ACL Message implementation complete")

    ACL Message implementation complete

In [ ]:
# Example: ACL message exchange with conversation tracking
tracker = ConversationTracker()

print("=== ACL Message Exchange ===")

# 1. Request message
request = ACLMessage(
    performative=Performative.REQUEST,
    sender="agent-001",
    receiver="agent-002",
    content={
        "action": "analyze_data",
        "parameters": {"dataset_id": "sales_2024"}
    },
    protocol="fipa-request",
    reply_by=(datetime.utcnow() + timedelta(seconds=30)).isoformat()
)
tracker.add_message(request)
print("\n1. REQUEST sent")
print(json.dumps(request.to_dict(), indent=2)[:300] + "...")

# 2. Agree to perform
agree = ACLMessage(
    performative=Performative.AGREE,
    sender="agent-002",
    receiver="agent-001",
    content={"message": "I will perform the analysis"},
    protocol="fipa-request",
    conversation_id=request.conversation_id,
    in_reply_to=request.reply_with
)
tracker.add_message(agree)
print(f"\n2. AGREE received (reply to: {request.reply_with[:8]}...)")

# 3. Inform result
result = ACLMessage(
    performative=Performative.INFORM_RESULT,
    sender="agent-002",
    receiver="agent-001",
    content={
        "status": "success",
        "result": {"trend": "upward", "confidence": 0.92}
    },
    protocol="fipa-request",
    conversation_id=request.conversation_id,
    in_reply_to=agree.reply_with
)
tracker.add_message(result)
print(f"\n3. INFORM_RESULT received")
print(f"   Content: {result.content}")

# View conversation thread
print("\n=== Conversation Thread ===")
thread = tracker.get_thread(result)
for i, msg in enumerate(thread, 1):
    print(f"{i}. {msg.performative.value.upper()} ({msg.sender} → {msg.receiver})")

# Check pending replies
print("\n=== Pending Replies ===")
pending = tracker.get_pending_replies("agent-001")
print(f"Agent-001 has {len(pending)} pending replies")

    === ACL Message Exchange ===

    1. REQUEST sent

    {

      "message_id": "c610e570-93a2-4751-998e-377fb44f8517",

      "performative": "request",

      "sender": "agent-001",

      "receiver": "agent-002",

      "content": {

        "action": "analyze_data",

        "parameters": {

          "dataset_id": "sales_2024"

        }

      },

      "language": "JSON",

      "ontology": "default",

      "pr...

    2. AGREE received (reply to: a1f7dddc...)

    3. INFORM_RESULT received

       Content: {'status': 'success', 'result': {'trend': 'upward', 'confidence': 0.92}}

    === Conversation Thread ===

    1. REQUEST (agent-001 → agent-002)

    2. AGREE (agent-002 → agent-001)

    3. INFORM-RESULT (agent-002 → agent-001)

    === Pending Replies ===

    Agent-001 has 0 pending replies

## 6. Collaboration: Contract Net Protocol (CNP)

### What is Contract Net Protocol?

Imagine you need to hire someone to paint your house. You would:

1. **Announce the job** (Call for Proposals)

2. **Collect bids** from painters (Proposals)

3. **Compare prices and quality** (Evaluation)

4. **Hire the best one** (Award Contract)

5. **Reject others** politely (Reject Proposals)

Contract Net Protocol does exactly this for agents!

### The 5-Step Process

In [ ]:
Step 1: MANAGER broadcasts "Call for Proposals (CFP)"
        ↓
Step 2: CONTRACTORS submit proposals (price, time, quality)
        ↓
Step 3: MANAGER evaluates all proposals
        ↓
Step 4: MANAGER accepts best proposal
        ↓
Step 5: MANAGER rejects other proposals

### Why Use CNP?

✅ **Fair Competition**: All capable agents can bid  

✅ **Optimal Selection**: Choose based on cost, time, quality  

✅ **Transparency**: Clear process for task allocation  

✅ **Scalability**: Works with 2 or 200 agents  

### Real-World Example

**Task**: "Analyze 1TB of customer data"

**Proposals Received**:

- Agent A: $100, 24 hours, 95% quality

- Agent B: $150, 12 hours, 99% quality ⭐ (WINNER - best quality/time)

- Agent C: $80, 48 hours, 90% quality

In [ ]:
@dataclass
class TaskSpecification:
    """Task specification for CNP"""
    task_id: str
    description: str
    requirements: Dict[str, Any]
    deadline: str
    evaluation_criteria: List[str] = field(default_factory=lambda: ["cost", "time", "quality"])

@dataclass
class Proposal:
    """Bid proposal from contractor"""
    contractor_id: str
    task_id: str
    cost: float
    completion_time: int  # hours
    quality_score: float  # 0-1
    capabilities: List[str]
    confidence: float = 0.9
    additional_terms: Dict[str, Any] = field(default_factory=dict)
    
    def score(self, weights: Dict[str, float] = None) -> float:
        """Calculate weighted score for proposal"""
        if weights is None:
            weights = {"cost": 0.4, "time": 0.3, "quality": 0.3}
        
        # Normalize metrics (lower is better for cost/time, higher for quality)
        cost_score = 1.0 / (1.0 + self.cost / 100.0)  # Normalize cost
        time_score = 1.0 / (1.0 + self.completion_time / 10.0)
        quality_score = self.quality_score
        
        return (weights["cost"] * cost_score +
                weights["time"] * time_score +
                weights["quality"] * quality_score)

class ContractNetManager:
    """Manager agent for Contract Net Protocol"""
    
    def __init__(self, manager_id: str, directory: DirectoryFacilitator):
        self.manager_id = manager_id
        self.directory = directory
        self.conversations = ConversationTracker()
        self._active_cfps: Dict[str, TaskSpecification] = {}
        self._proposals: Dict[str, List[Proposal]] = defaultdict(list)
    
    def call_for_proposals(self, task: TaskSpecification, 
                          criteria: SearchCriteria = None) -> str:
        """Broadcast CFP to capable agents"""
        conversation_id = str(uuid.uuid4())
        self._active_cfps[conversation_id] = task
        
        # Find capable agents
        if criteria is None:
            criteria = SearchCriteria(state=AgentState.ACTIVE)
        
        agents = self.directory.search(criteria)
        
        print(f"\n[CNP] Broadcasting CFP for task: {task.description}")
        print(f"      Found {len(agents)} potential contractors")
        
        # Send CFP messages
        for agent in agents:
            cfp_msg = ACLMessage(
                performative=Performative.CFP,
                sender=self.manager_id,
                receiver=agent.agent_id,
                content={
                    "task_id": task.task_id,
                    "description": task.description,
                    "requirements": task.requirements,
                    "deadline": task.deadline
                },
                protocol="fipa-contract-net",
                conversation_id=conversation_id,
                reply_by=task.deadline
            )
            self.conversations.add_message(cfp_msg)
            print(f"      → CFP sent to {agent.name}")
        
        return conversation_id
    
    def receive_proposal(self, proposal: Proposal, conversation_id: str):
        """Receive proposal from contractor"""
        self._proposals[conversation_id].append(proposal)
        print(f"\n[CNP] Proposal received from {proposal.contractor_id}")
        print(f"      Cost: ${proposal.cost}, Time: {proposal.completion_time}h, Quality: {proposal.quality_score}")
    
    def evaluate_proposals(self, conversation_id: str, 
                          weights: Dict[str, float] = None) -> Optional[Proposal]:
        """Evaluate and select best proposal"""
        proposals = self._proposals.get(conversation_id, [])
        
        if not proposals:
            print(f"\n[CNP] No proposals received")
            return None
        
        print(f"\n[CNP] Evaluating {len(proposals)} proposals...")
        
        # Score and rank
        scored = [(p, p.score(weights)) for p in proposals]
        scored.sort(key=lambda x: x[1], reverse=True)
        
        # Display ranking
        for i, (prop, score) in enumerate(scored, 1):
            print(f"      {i}. {prop.contractor_id}: score={score:.3f}")
        
        winner = scored[0][0]
        return winner
    
    def award_contract(self, conversation_id: str, winner: Proposal):
        """Award contract to winning proposal"""
        task = self._active_cfps[conversation_id]
        all_proposals = self._proposals[conversation_id]
        
        print(f"\n[CNP] Awarding contract for '{task.description}'")
        print(f"      Winner: {winner.contractor_id}")
        
        # Send ACCEPT to winner
        accept_msg = ACLMessage(
            performative=Performative.ACCEPT_PROPOSAL,
            sender=self.manager_id,
            receiver=winner.contractor_id,
            content={
                "task_id": task.task_id,
                "contract_terms": {
                    "cost": winner.cost,
                    "deadline": task.deadline
                }
            },
            protocol="fipa-contract-net",
            conversation_id=conversation_id
        )
        self.conversations.add_message(accept_msg)
        
        # Send REJECT to others
        for proposal in all_proposals:
            if proposal.contractor_id != winner.contractor_id:
                reject_msg = ACLMessage(
                    performative=Performative.REJECT_PROPOSAL,
                    sender=self.manager_id,
                    receiver=proposal.contractor_id,
                    content={
                        "task_id": task.task_id,
                        "reason": "Another proposal selected"
                    },
                    protocol="fipa-contract-net",
                    conversation_id=conversation_id
                )
                self.conversations.add_message(reject_msg)
                print(f"      Rejected: {proposal.contractor_id}")

class ContractorAgent:
    """Contractor agent for CNP"""
    
    def __init__(self, card: AgentCard, base_cost: float = 100.0):
        self.card = card
        self.base_cost = base_cost
        self.active_contracts: List[str] = []
    
    def evaluate_cfp(self, cfp_content: Dict[str, Any]) -> Optional[Proposal]:
        """Evaluate CFP and generate proposal"""
        requirements = cfp_content.get("requirements", {})
        required_skills = requirements.get("skills", [])
        
        # Check if we have required skills
        if not all(skill in self.card.skills for skill in required_skills):
            print(f"      [{self.card.name}] Cannot bid - missing skills")
            return None
        
        # Check availability
        if len(self.active_contracts) >= self.card.max_conversations:
            print(f"      [{self.card.name}] Cannot bid - at capacity")
            return None
        
        # Generate proposal
        workload_factor = 1.0 + (len(self.active_contracts) * 0.2)
        cost = self.base_cost * workload_factor
        completion_time = int(10 * workload_factor)
        
        proposal = Proposal(
            contractor_id=self.card.agent_id,
            task_id=cfp_content["task_id"],
            cost=cost,
            completion_time=completion_time,
            quality_score=self.card.qos.availability,
            capabilities=self.card.skills,
            confidence=0.85 if len(self.active_contracts) == 0 else 0.75
        )
        
        print(f"      [{self.card.name}] Submitting proposal: ${cost:.0f}, {completion_time}h")
        return proposal

print("Contract Net Protocol implementation complete")

    Contract Net Protocol implementation complete

In [ ]:
# Example: Complete Contract Net Protocol execution
df = DirectoryFacilitator()

# Register contractor agents
contractors = [
    ContractorAgent(
        AgentCard(
            agent_id="contractor-001",
            name="FastProcessor",
            agent_type="processor",
            skills=["data_processing", "analysis"],
            state=AgentState.ACTIVE,
            qos=QoSParameters(availability=0.95)
        ),
        base_cost=80.0
    ),
    ContractorAgent(
        AgentCard(
            agent_id="contractor-002",
            name="QualityAnalyzer",
            agent_type="analyzer",
            skills=["data_processing", "analysis", "validation"],
            state=AgentState.ACTIVE,
            qos=QoSParameters(availability=0.99)
        ),
        base_cost=120.0
    ),
    ContractorAgent(
        AgentCard(
            agent_id="contractor-003",
            name="BudgetProcessor",
            agent_type="processor",
            skills=["data_processing"],
            state=AgentState.ACTIVE,
            qos=QoSParameters(availability=0.90)
        ),
        base_cost=60.0
    )
]

# Register in directory
for contractor in contractors:
    df.register(contractor.card)

# Create manager
manager = ContractNetManager("manager-001", df)

# Define task
task = TaskSpecification(
    task_id="task-001",
    description="Process and analyze customer data",
    requirements={
        "skills": ["data_processing", "analysis"],
        "min_quality": 0.9
    },
    deadline=(datetime.utcnow() + timedelta(hours=24)).isoformat()
)

print("=== Contract Net Protocol Demo ===")

# Phase 1: Call for Proposals
conv_id = manager.call_for_proposals(
    task,
    SearchCriteria(skills=["data_processing"], state=AgentState.ACTIVE)
)

# Phase 2: Contractors submit proposals
cfp_content = {
    "task_id": task.task_id,
    "description": task.description,
    "requirements": task.requirements,
    "deadline": task.deadline
}

for contractor in contractors:
    proposal = contractor.evaluate_cfp(cfp_content)
    if proposal:
        manager.receive_proposal(proposal, conv_id)

# Phase 3: Evaluate and award
winner = manager.evaluate_proposals(conv_id, weights={"cost": 0.3, "time": 0.2, "quality": 0.5})

if winner:
    manager.award_contract(conv_id, winner)

print("\n=== CNP Complete ===")

    === Contract Net Protocol Demo ===

    [CNP] Broadcasting CFP for task: Process and analyze customer data

          Found 0 potential contractors

          [FastProcessor] Submitting proposal: $80, 10h

    [CNP] Proposal received from contractor-001

          Cost: $80.0, Time: 10h, Quality: 0.95

          [QualityAnalyzer] Submitting proposal: $120, 10h

    [CNP] Proposal received from contractor-002

          Cost: $120.0, Time: 10h, Quality: 0.99

          [BudgetProcessor] Cannot bid - missing skills

    [CNP] Evaluating 2 proposals...

          1. contractor-001: score=0.742

          2. contractor-002: score=0.731

    [CNP] Awarding contract for 'Process and analyze customer data'

          Winner: contractor-001

          Rejected: contractor-002

    === CNP Complete ===

## 7. Production Features

### 7.1 Retry with Exponential Backoff

In [ ]:
import time
import random
from typing import Callable, TypeVar

T = TypeVar('T')

class RetryPolicy:
    """Configurable retry with exponential backoff"""
    
    def __init__(self, 
                 max_attempts: int = 3,
                 base_delay: float = 1.0,
                 max_delay: float = 60.0,
                 exponential_base: float = 2.0,
                 jitter: bool = True):
        self.max_attempts = max_attempts
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.exponential_base = exponential_base
        self.jitter = jitter
    
    def execute(self, func: Callable[[], T], 
                on_retry: Callable[[int, Exception], None] = None) -> T:
        """Execute function with retry"""
        last_exception = None
        
        for attempt in range(1, self.max_attempts + 1):
            try:
                return func()
            except Exception as e:
                last_exception = e
                
                if attempt == self.max_attempts:
                    break
                
                # Calculate delay
                delay = min(
                    self.base_delay * (self.exponential_base ** (attempt - 1)),
                    self.max_delay
                )
                
                # Add jitter
                if self.jitter:
                    delay *= (0.5 + random.random() * 0.5)
                
                if on_retry:
                    on_retry(attempt, e)
                
                print(f"  Retry {attempt}/{self.max_attempts} after {delay:.2f}s: {e}")
                time.sleep(delay)
        
        raise last_exception

# Example usage
print("=== Retry with Exponential Backoff ===")

call_count = 0
def flaky_operation():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError(f"Attempt {call_count} failed")
    return "Success!"

retry_policy = RetryPolicy(max_attempts=5, base_delay=0.5)
result = retry_policy.execute(flaky_operation)
print(f"\nResult: {result}")

    === Retry with Exponential Backoff ===

      Retry 1/5 after 0.30s: Attempt 1 failed

      Retry 2/5 after 0.99s: Attempt 2 failed

    Result: Success!

### 7.2 Circuit Breaker Pattern

In [ ]:
from enum import Enum
from datetime import datetime, timedelta

class CircuitState(Enum):
    CLOSED = "closed"      # Normal operation
    OPEN = "open"          # Failing, reject requests
    HALF_OPEN = "half_open"  # Testing recovery

class CircuitBreaker:
    """Prevent cascading failures"""
    
    def __init__(self,
                 failure_threshold: int = 5,
                 success_threshold: int = 2,
                 timeout: float = 60.0):
        self.failure_threshold = failure_threshold
        self.success_threshold = success_threshold
        self.timeout = timeout
        
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.success_count = 0
        self.last_failure_time = None
    
    def call(self, func: Callable[[], T]) -> T:
        """Execute function through circuit breaker"""
        if self.state == CircuitState.OPEN:
            if self._should_attempt_reset():
                self.state = CircuitState.HALF_OPEN
                print(f"  [Circuit] Attempting recovery (HALF_OPEN)")
            else:
                raise Exception(f"Circuit breaker OPEN - failing fast")
        
        try:
            result = func()
            self._on_success()
            return result
        except Exception as e:
            self._on_failure()
            raise
    
    def _should_attempt_reset(self) -> bool:
        """Check if enough time has passed to attempt reset"""
        if not self.last_failure_time:
            return True
        elapsed = (datetime.utcnow() - self.last_failure_time).total_seconds()
        return elapsed >= self.timeout
    
    def _on_success(self):
        """Handle successful call"""
        self.failure_count = 0
        
        if self.state == CircuitState.HALF_OPEN:
            self.success_count += 1
            if self.success_count >= self.success_threshold:
                self.state = CircuitState.CLOSED
                self.success_count = 0
                print(f"  [Circuit] Recovered (CLOSED)")
    
    def _on_failure(self):
        """Handle failed call"""
        self.failure_count += 1
        self.last_failure_time = datetime.utcnow()
        self.success_count = 0
        
        if self.failure_count >= self.failure_threshold:
            if self.state != CircuitState.OPEN:
                self.state = CircuitState.OPEN
                print(f"  [Circuit] Opened after {self.failure_count} failures")

# Example
print("\n=== Circuit Breaker Demo ===")
breaker = CircuitBreaker(failure_threshold=3, timeout=2.0)

attempt = 0
def unreliable_service():
    global attempt
    attempt += 1
    if attempt <= 5:  # Fail first 5 times
        raise Exception(f"Service error (attempt {attempt})")
    return "OK"

# Test circuit breaker
for i in range(10):
    try:
        result = breaker.call(unreliable_service)
        print(f"  Call {i+1}: {result} [State: {breaker.state.value}]")
    except Exception as e:
        print(f"  Call {i+1}: Failed - {e} [State: {breaker.state.value}]")
    
    if i == 4:  # Wait for timeout after circuit opens
        print("  Waiting for timeout...")
        time.sleep(2.1)

    === Circuit Breaker Demo ===

      Call 1: Failed - Service error (attempt 1) [State: closed]

      Call 2: Failed - Service error (attempt 2) [State: closed]

      [Circuit] Opened after 3 failures

      Call 3: Failed - Service error (attempt 3) [State: open]

      Call 4: Failed - Circuit breaker OPEN - failing fast [State: open]

      Call 5: Failed - Circuit breaker OPEN - failing fast [State: open]

      Waiting for timeout...

      [Circuit] Attempting recovery (HALF_OPEN)

      [Circuit] Opened after 4 failures

      Call 6: Failed - Service error (attempt 4) [State: open]

      Call 7: Failed - Circuit breaker OPEN - failing fast [State: open]

      Call 8: Failed - Circuit breaker OPEN - failing fast [State: open]

      Call 9: Failed - Circuit breaker OPEN - failing fast [State: open]

      Call 10: Failed - Circuit breaker OPEN - failing fast [State: open]

## 8. Complete Working System

Putting it all together: Full multi-agent system with discovery, lifecycle, messaging, and collaboration.

In [ ]:
# Complete A2A System Demo
print("="*60)
print("COMPLETE A2A MULTI-AGENT SYSTEM")
print("="*60)

# 1. Initialize infrastructure
print("\n[PHASE 1] Infrastructure Setup")
directory = DirectoryFacilitator()
lifecycle_mgr = LifecycleManager(directory)
skill_reg = SkillRegistry()

# Register skills
for skill in [data_cleaning, feature_engineering, model_training]:
    skill_reg.register(skill)
print(f"  ✓ Registered {len(skill_reg._skills)} skills")

# 2. Create and register agents
print("\n[PHASE 2] Agent Registration")
agents_to_create = [
    {
        "name": "DataPreprocessor",
        "type": "processor",
        "skills": ["data_cleaning"],
        "cost": 50.0
    },
    {
        "name": "FeatureEngineer",
        "type": "engineer",
        "skills": ["feature_engineering"],
        "cost": 75.0
    },
    {
        "name": "MLTrainer",
        "type": "ml-trainer",
        "skills": ["model_training"],
        "cost": 150.0
    }
]

agent_objects = []
for spec in agents_to_create:
    card = AgentCard(
        agent_id=f"agent-{spec['name'].lower()}",
        name=spec["name"],
        agent_type=spec["type"],
        skills=spec["skills"],
        endpoints={"http": f"http://localhost:800{len(agent_objects)+1}"}
    )
    
    lifecycle_mgr.bootstrap(card)
    lifecycle_mgr.activate(card.agent_id)
    
    agent_obj = ContractorAgent(card, base_cost=spec["cost"])
    agent_objects.append(agent_obj)
    
    print(f"  ✓ {card.name} [Skills: {', '.join(card.skills)}]")

# 3. Task allocation via CNP
print("\n[PHASE 3] Task Allocation (Contract Net)")
manager = ContractNetManager("orchestrator-001", directory)

tasks = [
    TaskSpecification(
        task_id="t1",
        description="Clean raw customer data",
        requirements={"skills": ["data_cleaning"]},
        deadline=(datetime.utcnow() + timedelta(hours=2)).isoformat()
    ),
    TaskSpecification(
        task_id="t2",
        description="Engineer features for ML",
        requirements={"skills": ["feature_engineering"]},
        deadline=(datetime.utcnow() + timedelta(hours=4)).isoformat()
    ),
    TaskSpecification(
        task_id="t3",
        description="Train predictive model",
        requirements={"skills": ["model_training"]},
        deadline=(datetime.utcnow() + timedelta(hours=8)).isoformat()
    )
]

awarded_contracts = []
for task in tasks:
    print(f"\nTask: {task.description}")
    conv_id = manager.call_for_proposals(task)
    
    # Collect proposals
    cfp_content = {
        "task_id": task.task_id,
        "description": task.description,
        "requirements": task.requirements,
        "deadline": task.deadline
    }
    
    for agent in agent_objects:
        proposal = agent.evaluate_cfp(cfp_content)
        if proposal:
            manager.receive_proposal(proposal, conv_id)
    
    # Award
    winner = manager.evaluate_proposals(conv_id)
    if winner:
        manager.award_contract(conv_id, winner)
        awarded_contracts.append((task, winner))

# 4. System statistics
print("\n" + "="*60)
print("[FINAL STATISTICS]")
print("="*60)

stats = directory.get_statistics()
print(f"\nDirectory:")
print(f"  Total Agents: {stats['total_agents']}")
print(f"  Active: {stats['by_state'].get('active', 0)}")
print(f"  Skills Available: {stats['unique_skills']}")

print(f"\nContracts Awarded: {len(awarded_contracts)}")
for task, winner in awarded_contracts:
    print(f"  • {task.task_id}: {winner.contractor_id} (${winner.cost:.0f}, {winner.completion_time}h)")

total_cost = sum(w.cost for _, w in awarded_contracts)
print(f"\nTotal Project Cost: ${total_cost:.2f}")

print("\n" + "="*60)
print("SYSTEM OPERATIONAL")
print("="*60)

    ============================================================

    COMPLETE A2A MULTI-AGENT SYSTEM

    ============================================================

    [PHASE 1] Infrastructure Setup

      ✓ Registered 3 skills

    [PHASE 2] Agent Registration

    [BOOTSTRAP] DataPreprocessor initialized

    [LIFECYCLE] DataPreprocessor: registered -> active

      ✓ DataPreprocessor [Skills: data_cleaning]

    [BOOTSTRAP] FeatureEngineer initialized

    [LIFECYCLE] FeatureEngineer: registered -> active

      ✓ FeatureEngineer [Skills: feature_engineering]

    [BOOTSTRAP] MLTrainer initialized

    [LIFECYCLE] MLTrainer: registered -> active

      ✓ MLTrainer [Skills: model_training]

    [PHASE 3] Task Allocation (Contract Net)

    Task: Clean raw customer data

    [CNP] Broadcasting CFP for task: Clean raw customer data

          Found 3 potential contractors

          → CFP sent to MLTrainer

          → CFP sent to DataPreprocessor

          → CFP sent to FeatureEngineer

          [DataPreprocessor] Submitting proposal: $50, 10h

    [CNP] Proposal received from agent-datapreprocessor

          Cost: $50.0, Time: 10h, Quality: 0.99

          [FeatureEngineer] Cannot bid - missing skills

          [MLTrainer] Cannot bid - missing skills

    [CNP] Evaluating 1 proposals...

          1. agent-datapreprocessor: score=0.714

    [CNP] Awarding contract for 'Clean raw customer data'

          Winner: agent-datapreprocessor

    Task: Engineer features for ML

    [CNP] Broadcasting CFP for task: Engineer features for ML

          Found 3 potential contractors

          → CFP sent to MLTrainer

          → CFP sent to DataPreprocessor

          → CFP sent to FeatureEngineer

          [DataPreprocessor] Cannot bid - missing skills

          [FeatureEngineer] Submitting proposal: $75, 10h

    [CNP] Proposal received from agent-featureengineer

          Cost: $75.0, Time: 10h, Quality: 0.99

          [MLTrainer] Cannot bid - missing skills

    [CNP] Evaluating 1 proposals...

          1. agent-featureengineer: score=0.676

    [CNP] Awarding contract for 'Engineer features for ML'

          Winner: agent-featureengineer

    Task: Train predictive model

    [CNP] Broadcasting CFP for task: Train predictive model

          Found 3 potential contractors

          → CFP sent to MLTrainer

          → CFP sent to DataPreprocessor

          → CFP sent to FeatureEngineer

          [DataPreprocessor] Cannot bid - missing skills

          [FeatureEngineer] Cannot bid - missing skills

          [MLTrainer] Submitting proposal: $150, 10h

    [CNP] Proposal received from agent-mltrainer

          Cost: $150.0, Time: 10h, Quality: 0.99

    [CNP] Evaluating 1 proposals...

          1. agent-mltrainer: score=0.607

    [CNP] Awarding contract for 'Train predictive model'

          Winner: agent-mltrainer

    ============================================================

    [FINAL STATISTICS]

    ============================================================

    Directory:

      Total Agents: 3

      Active: 3

      Skills Available: 3

    Contracts Awarded: 3

      • t1: agent-datapreprocessor ($50, 10h)

      • t2: agent-featureengineer ($75, 10h)

      • t3: agent-mltrainer ($150, 10h)

    Total Project Cost: $275.00

    ============================================================

    SYSTEM OPERATIONAL

    ============================================================

## 📚 Summary: A2A Protocol for Beginners

Congratulations! You've learned the complete A2A (Agent-to-Agent) Protocol. Let's recap the key concepts:

### 🎯 What is A2A?

A2A is an **open protocol** (like HTTP for the web) that allows AI agents from different companies and frameworks to work together seamlessly.

### 🔑 Key Components You Learned

#### 1. **Agent Cards** - Business Cards for Agents

- Contains agent identity, skills, and contact information

- Enables automatic discovery

- **Real-world analogy**: Like a LinkedIn profile

#### 2. **Directory Facilitator** - The Yellow Pages

- Central registry where agents register

- Fast search using indices (skill, type, availability)

- Event notifications (agent joins/leaves)

- **Real-world analogy**: Like Google for finding agents

#### 3. **Skills** - What Agents Can Do

- Defines capabilities with inputs/outputs

- Enables skill composition (chaining)

- Searchable by category and keywords

- **Real-world analogy**: Like job requirements on a resume

#### 4. **Lifecycle Management** - Agent Health & States

- 7 states: Initiated → Registered → Active → Suspended → Migrating → Terminating → Terminated

- Heartbeat monitoring for health checks

- Automatic recovery mechanisms

- **Real-world analogy**: Like employee lifecycle (hiring to retirement)

#### 5. **ACL Messages** - Agent Communication

- Structured messages with performatives (REQUEST, INFORM, AGREE, etc.)

- Conversation threading

- Reply management and deadlines

- **Real-world analogy**: Like business emails with standard formats

#### 6. **Contract Net Protocol** - Task Allocation

- Competitive bidding for tasks

- Fair evaluation (cost, time, quality)

- Transparent selection process

- **Real-world analogy**: Like hiring contractors through bidding

#### 7. **Production Patterns** - Reliability

- Retry with exponential backoff (handle temporary failures)

- Circuit breaker (prevent cascading failures)

- **Real-world analogy**: Like safety mechanisms in buildings

### 🚀 What Can You Build?

With A2A, you can create:

- **Multi-agent systems** where agents collaborate on complex tasks

- **Distributed workflows** across different organizations

- **Self-organizing systems** that discover and coordinate automatically

- **Resilient applications** with automatic failure recovery

### 📖 Quick Reference

| Component | Purpose | Key Method |

|-----------|---------|-----------|

| AgentCard | Identity | Define who you are |

| DirectoryFacilitator | Discovery | `register()`, `search()` |

| SkillRegistry | Capabilities | `register()`, `search_by_keywords()` |

| LifecycleManager | Health | `activate()`, `suspend()`, `heartbeat()` |

| ACLMessage | Communication | Create messages with performatives |

| ContractNetManager | Collaboration | `call_for_proposals()`, `award_contract()` |

### 🎓 Next Steps

1. **Experiment**: Modify the code examples to create your own agents

2. **Build**: Create a small multi-agent system (e.g., data pipeline)

3. **Explore**: Check Google's A2A SDK documentation for advanced features

4. **Scale**: Deploy agents across different servers and frameworks

### 💡 Key Takeaways

✅ A2A enables **interoperability** - agents from different frameworks can work together  

✅ **Discovery** is automatic through the Directory Facilitator  

✅ **Communication** is standardized using ACL messages  

✅ **Collaboration** happens through protocols like Contract Net  

✅ **Reliability** is built-in with retry, recovery, and circuit breakers  

### 🔗 Resources

- **Official A2A Site**: https://a2a.dev

- **Google A2A SDK**: `pip install a2a-sdk`

- **Linux Foundation**: A2A is open-source under Apache 2.0

- **Documentation**: https://docs.a2a.dev

---

**Remember**: A2A is like building with LEGO blocks - each component (agent) is independent, but they all fit together perfectly through the standardized protocol! 🧩

Happy agent building! 🤖✨